# FPL Data Analysis

Exploratory analysis of the 2024/25 FPL season data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Load data
players = pd.read_csv('../data/raw/players.csv')
features = pd.read_csv('../data/processed/features.csv')
teams = pd.read_csv('../data/raw/teams.csv')
team_map = dict(zip(teams['id'], teams['short_name']))

print(f'Players: {len(players)}')
print(f'Feature rows: {len(features)}')
print(f'GW range: {features["round"].min()} - {features["round"].max()}')

## Top Scorers by Position

In [ ]:
season_totals = features.groupby(['player_id', 'web_name', 'position']).agg(
    total_pts=('total_points', 'sum'),
    games=('round', 'count'),
    avg_pts=('total_points', 'mean'),
).reset_index().sort_values('total_pts', ascending=False)

for pos in ['GK', 'DEF', 'MID', 'FWD']:
    print(f'\nTop 5 {pos}:')
    display(season_totals[season_totals['position'] == pos].head())

## Points Distribution by Position

In [ ]:
pos_colors = {'GK': '#FFD700', 'DEF': '#2ECC71', 'MID': '#3498DB', 'FWD': '#E74C3C'}

fig, ax = plt.subplots(figsize=(10, 5))
for pos in ['GK', 'DEF', 'MID', 'FWD']:
    data = features[features['position'] == pos]['total_points']
    ax.hist(data, bins=range(-5, 25), alpha=0.5, label=pos, color=pos_colors[pos])
ax.set_xlabel('Gameweek Points')
ax.set_ylabel('Frequency')
ax.set_title('Points Distribution by Position')
ax.legend()
plt.tight_layout()
plt.show()

## Feature Correlations with Points

In [ ]:
numeric_cols = features.select_dtypes(include=[np.number]).columns
corr_with_pts = features[numeric_cols].corr()['total_points'].sort_values(ascending=False)
print('Correlation with total_points:')
print(corr_with_pts.to_string())

## Home vs Away Performance

In [ ]:
home_away = features.groupby(['position', 'is_home'])['total_points'].mean().unstack()
home_away.columns = ['Away', 'Home']
home_away.plot(kind='bar', color=['#E74C3C', '#2ECC71'], figsize=(8, 4))
plt.title('Average Points: Home vs Away by Position')
plt.ylabel('Avg Points')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Backtest Results

In [ ]:
import os
bt_path = '../data/processed/backtest_results.csv'
if os.path.exists(bt_path):
    bt = pd.read_csv(bt_path)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(bt['gameweek'], bt['cumulative_actual'], 'o-', label='Model', color='#2ECC71')
    if 'cumulative_avg_manager' in bt.columns:
        ax.plot(bt['gameweek'], bt['cumulative_avg_manager'], 's--', label='Avg Manager', color='#95A5A6')
    ax.set_xlabel('Gameweek')
    ax.set_ylabel('Cumulative Points')
    ax.set_title('Backtest: Model vs Average Manager')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    display(bt)
else:
    print('No backtest results yet. Run: python -m src.model')